In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:
df = pd.read_csv("../data/processed/cleaned_data.csv")

In [3]:
df.head()

,order_date,ship_date,ship_mode,segment,country,city,state,postal_code,region,category,sub-category,sales,quantity,discount,profit
0,2016-11-08,2016-11-11,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0.00,41.9136
1,2016-11-08,2016-11-11,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0.00,219.5820
2,2016-06-12,2016-06-16,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0.00,6.8714
3,2015-10-11,2015-10-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.0310
4,2015-10-11,2015-10-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,0.20,2.5164


In [4]:
df.category.unique()

array(['Furniture', 'Office Supplies', 'Technology'], dtype=object)

In [5]:
df.region.unique()

array(['South', 'West', 'Central', 'East'], dtype=object)

In [6]:
df.segment.unique()

array(['Consumer', 'Corporate', 'Home Office'], dtype=object)

### one hot encoding category, region and segment

In [7]:
nominal_col=['category','region','segment']

In [8]:
cols_to_encode = [col for col in nominal_col if col in df.columns]
df = pd.get_dummies(df,columns=nominal_col,drop_first=True,dtype=int)


In [9]:
df.head()

,order_date,ship_date,ship_mode,country,city,state,postal_code,sub-category,sales,quantity,discount,profit,category_Office Supplies,category_Technology,region_East,region_South,region_West,segment_Corporate,segment_Home Office
0,2016-11-08,2016-11-11,Second Class,United States,Henderson,Kentucky,42420,Bookcases,261.9600,2,0.00,41.9136,0,0,0,1,0,0,0
1,2016-11-08,2016-11-11,Second Class,United States,Henderson,Kentucky,42420,Chairs,731.9400,3,0.00,219.5820,0,0,0,1,0,0,0
2,2016-06-12,2016-06-16,Second Class,United States,Los Angeles,California,90036,Labels,14.6200,2,0.00,6.8714,1,0,0,0,1,1,0
3,2015-10-11,2015-10-18,Standard Class,United States,Fort Lauderdale,Florida,33311,Tables,957.5775,5,0.45,-383.0310,0,0,0,1,0,0,0
4,2015-10-11,2015-10-18,Standard Class,United States,Fort Lauderdale,Florida,33311,Storage,22.3680,2,0.20,2.5164,1,0,0,1,0,0,0


### Dropping country and postal code

In [10]:
df.drop(columns=['country','postal_code'], inplace=True)

### Checking columns which are string

In [11]:
df.select_dtypes(include='object').columns

Index(['order_date', 'ship_date', 'ship_mode', 'city', 'state',
       'sub-category'],
      dtype='object')

In [12]:
df.drop(columns=['city'], inplace=True)

### Encoding ship_mode

In [13]:
nominal_cols = ["ship_mode"]

df = pd.get_dummies(
    df,
    columns=nominal_cols,
    drop_first=True,
    dtype=int
)

In [14]:
df.drop(columns=['state','sub-category'], inplace=True)

### Dropping order_date and ship_date

In [15]:
df.drop(columns=["order_date", "ship_date"], inplace=True)

### Checking if any string column left

In [16]:
df.select_dtypes(include='object').columns

Index([], dtype='object')

### Creating a derived feature profit margin

In [17]:
df["profit_margin"] = df["profit"] / df["sales"]

In [19]:
df.columns.tolist()

['sales',
 'quantity',
 'discount',
 'profit',
 'category_Office Supplies',
 'category_Technology',
 'region_East',
 'region_South',
 'region_West',
 'segment_Corporate',
 'segment_Home Office',
 'ship_mode_Same Day',
 'ship_mode_Second Class',
 'ship_mode_Standard Class',
 'profit_margin']

### Calculating VIF

In [18]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [20]:
X = df.drop(columns=["profit", "profit_margin"])

# Calculate VIF
vif_df = pd.DataFrame()
vif_df["Feature"] = X.columns
vif_df["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

# Sort by highest VIF
vif_df = vif_df.sort_values(by="VIF", ascending=False)

vif_df

,Feature,VIF
1,quantity,3.583079
12,ship_mode_Standard Class,3.512451
3,category_Office Supplies,3.189365
7,region_West,2.035304
5,region_East,1.888863
11,ship_mode_Second Class,1.796382
4,category_Technology,1.645761
8,segment_Corporate,1.530232
6,region_South,1.527897
2,discount,1.519707


In [21]:
df.head()

,sales,quantity,discount,profit,category_Office Supplies,category_Technology,region_East,region_South,region_West,segment_Corporate,segment_Home Office,ship_mode_Same Day,ship_mode_Second Class,ship_mode_Standard Class,profit_margin
0,261.9600,2,0.00,41.9136,0,0,0,1,0,0,0,0,1,0,0.1600
1,731.9400,3,0.00,219.5820,0,0,0,1,0,0,0,0,1,0,0.3000
2,14.6200,2,0.00,6.8714,1,0,0,0,1,1,0,0,1,0,0.4700
3,957.5775,5,0.45,-383.0310,0,0,0,1,0,0,0,0,0,1,-0.4000
4,22.3680,2,0.20,2.5164,1,0,0,1,0,0,0,0,0,1,0.1125


### Scaling sales, quantity and discount 

In [28]:
from sklearn.preprocessing import StandardScaler

# Columns to scale
numeric_cols = ["sales", "quantity", "discount"]

# Create a copy of the dataframe
df_scaled = df.copy()

# Initialize the scaler
scaler = StandardScaler()

# Scale only the numerical features
df_scaled[numeric_cols] = scaler.fit_transform(df_scaled[numeric_cols])

# Display first few rows
df_scaled.head()

,sales,quantity,discount,profit,category_Office Supplies,category_Technology,region_East,region_South,region_West,segment_Corporate,segment_Home Office,ship_mode_Same Day,ship_mode_Second Class,ship_mode_Standard Class,profit_margin
0,0.051510,-0.804303,-0.756643,41.9136,0,0,0,1,0,0,0,0,1,0,0.1600
1,0.805633,-0.354865,-0.756643,219.5820,0,0,0,1,0,0,0,0,1,0,0.3000
2,-0.345368,-0.804303,-0.756643,6.8714,1,0,0,0,1,1,0,0,1,0,0.4700
3,1.167688,0.544012,1.423149,-383.0310,0,0,0,1,0,0,0,0,0,1,-0.4000
4,-0.332935,-0.804303,0.212153,2.5164,1,0,0,1,0,0,0,0,0,1,0.1125


### Train-test-split

In [23]:
from sklearn.model_selection import train_test_split

# ============================================================
# Unscaled Dataset (Decision Tree & Random Forest)
# ============================================================

X = df.drop(columns=["profit", "profit_margin"])
y = df["profit"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ============================================================
# Scaled Dataset (Linear Regression)
# ============================================================

X_scaled = df_scaled.drop(columns=["profit", "profit_margin"])
y_lr = df_scaled["profit"]   # Same as df["profit"], not scaled

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_scaled,
    y_lr,
    test_size=0.2,
    random_state=42
)

# ============================================================
# Check Shapes
# ============================================================

print("Decision Tree / Random Forest")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nLinear Regression")
print("X_train_lr:", X_train_lr.shape)
print("X_test_lr :", X_test_lr.shape)
print("y_train_lr:", y_train_lr.shape)
print("y_test_lr :", y_test_lr.shape)

Decision Tree / Random Forest
X_train: (7995, 13)
X_test : (1999, 13)
y_train: (7995,)
y_test : (1999,)

Linear Regression
X_train_lr: (7995, 13)
X_test_lr : (1999, 13)
y_train_lr: (7995,)
y_test_lr : (1999,)


In [26]:
X_scaled.describe()

,sales,quantity,discount,category_Office Supplies,category_Technology,region_East,region_South,region_West,segment_Corporate,segment_Home Office,ship_mode_Same Day,ship_mode_Second Class,ship_mode_Standard Class
count,9.994000e+03,9.994000e+03,9.994000e+03,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,-4.550204e-17,4.870140e-17,7.251887e-17,0.602962,0.184811,0.284971,0.162097,0.320492,0.302181,0.178407,0.054333,0.194617,0.597158
std,1.000050e+00,1.000050e+00,1.000050e+00,0.489309,0.388163,0.451423,0.368558,0.466689,0.459226,0.382874,0.226684,0.395925,0.490494
min,-3.681144e-01,-1.253742e+00,-7.566435e-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-3.410996e-01,-8.043034e-01,-7.566435e-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-2.813929e-01,-3.548649e-01,2.121533e-01,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,-3.196013e-02,5.440122e-01,2.121533e-01,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000
max,3.595655e+01,4.588959e+00,3.118544e+00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


### Saving the train-test-split dataset of both scaled and unscaled data

In [29]:
# Save unscaled datasets (Decision Tree & Random Forest)

X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)


# Save scaled datasets (Linear Regression)

X_train_lr.to_csv("../data/processed/X_train_lr.csv", index=False)
X_test_lr.to_csv("../data/processed/X_test_lr.csv", index=False)

y_train_lr.to_csv("../data/processed/y_train_lr.csv", index=False)
y_test_lr.to_csv("../data/processed/y_test_lr.csv", index=False)

print("All datasets saved successfully!")

All datasets saved successfully!
